In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import false_discovery_control

from scipy.stats import chi2
from snp_analysis_tools_sherlock import *
from scipy.stats import binom
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot
from glob import glob

hv.extension('bokeh')

In [ ]:
assembly_metadata = pd.read_csv('assembly_glycerol_metadata_with_redo.csv')

In [ ]:
fnames = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/calculateFixedDiffsFastv3/*/*_fixed_diffs.csv')
all_dfs = []
for fname in fnames:
    df = pd.read_csv(fname).drop(columns='Unnamed: 0').rename(columns = {'index': 'sample2'})
    good_samples = df.loc[df['sample2'].isin(assembly_metadata['sample']),:]
    good_samples = good_samples.loc[good_samples['sample1'].isin(assembly_metadata['sample']),:]
  #  good_samples['parents1'] = good_samples['sample1'].transform(lambda x: e003_metadata.loc[e003_metadata['sample']==x,'parent_subjects'].values[0])
   # good_samples['parents2'] = good_samples['sample2'].transform(lambda x: e003_metadata.loc[e003_metadata['sample']==x,
    #                                                         'parent_subjects'].values[0])
    
    all_dfs.append(good_samples)
all_dfs = pd.concat(all_dfs)
good_samples = all_dfs
all_dfs.head()

In [ ]:
good_samples = all_dfs.loc[all_dfs['sample2'].isin(assembly_metadata['sample']),:]
good_samples = good_samples.loc[good_samples['sample1'].isin(assembly_metadata['sample']),:]
good_samples['parents1'] = good_samples['sample1'].transform(lambda x: assembly_metadata.loc[assembly_metadata['sample']==x,
                                                             'subject'].values[0])
good_samples['parents2'] = good_samples['sample2'].transform(lambda x: assembly_metadata.loc[assembly_metadata['sample']==x,
                                                             'subject'].values[0])


good_samples['media1'] = good_samples['sample1'].transform(lambda x: assembly_metadata.loc[assembly_metadata['sample']==x,
                                                             'media'].values[0])
good_samples['media2'] = good_samples['sample2'].transform(lambda x: assembly_metadata.loc[assembly_metadata['sample']==x,
                                                             'media'].values[0])

good_samples['type_meso_1'] = good_samples['parents1'] + '-' + good_samples['media1']
good_samples['type_meso_2'] = good_samples['parents2'] + '-' + good_samples['media2']
good_samples = good_samples.loc[good_samples['sample1']!=good_samples['sample2'],:]
good_samples =good_samples.loc[good_samples['parents1']!='AC/PP',:]
good_samples =good_samples.loc[good_samples['parents2']!='AC/PP',:]
good_samples['parents1'].unique()

In [ ]:
good_samples['log_fixed_diffs']= np.log10(1+good_samples['fixed_diffs'])
p.ray(x=2,angle=np.pi/2)
good_samples['same_meso']=good_samples['type_meso_1']==good_samples['type_meso_2']
good_samples['same_host']=good_samples['parents1']==good_samples['parents2']
good_samples_same_host=good_samples #.loc[good_samples['parents1']==good_samples['parents2'],:]

#good_samples_same_host=good_samples_same_host.loc[good_samples_same_host['type_meso_1']
p = iqplot.strip(good_samples_same_host,q='log_fixed_diffs',cats=['type_meso_1','type_meso_2'],color_column='same_host',
                                                  height=1000,jitter=True)

In [ ]:
bokeh.io.show(p)